In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!unzip -oq "/content/drive/MyDrive/Face_Mask_Project/Dataset_Mask.zip" -d /content/dataset
!pip install opencv-python matplotlib seaborn pillow tqdm -q

In [ ]:
import os
from collections import Counter
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

CLASS_NAMES = {
    0: 'mask_weared_incorrect',
    1: 'with_mask',
    2: 'without_mask'
}
COLORS = ['#e74c3c', '#2ecc71', '#e67e22']

data_dir = '/content/dataset'
splits   = ['train', 'val', 'test']

print("=" * 55)
print("PHÂN TÍCH PHÂN PHỐI DỮ LIỆU")
print("=" * 55)

all_stats = {}
for split in splits:
    split_path = os.path.join(data_dir, split)
    if not os.path.exists(split_path):
        print(f"⚠️  Không tìm thấy thư mục: {split_path}")
        continue
    classes = sorted(os.listdir(split_path))
    counts  = {}
    for cls in classes:
        cls_path = os.path.join(split_path, cls)
        if os.path.isdir(cls_path):
            counts[cls] = len([
                f for f in os.listdir(cls_path)
                if f.lower().endswith(('.jpg', '.jpeg', '.png'))
            ])
    all_stats[split] = counts
    total = sum(counts.values())
    print(f"\n📂 {split.upper()} — Tổng: {total} ảnh")
    for cls, cnt in counts.items():
        pct = cnt / total * 100 if total > 0 else 0
        print(f"   {cls:<35} {cnt:>5} ảnh ({pct:.1f}%)")

# Vẽ biểu đồ
fig, axes = plt.subplots(1, len(all_stats), figsize=(16, 5))
fig.suptitle('Phân phối dữ liệu theo tập', fontsize=15, fontweight='bold')

for ax, (split, counts) in zip(axes, all_stats.items()):
    labels = list(counts.keys())
    values = list(counts.values())
    bars = ax.bar(range(len(labels)), values,
                  color=COLORS[:len(labels)], alpha=0.85, edgecolor='white')
    ax.set_title(split.upper(), fontweight='bold')
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=25, ha='right', fontsize=9)
    ax.set_ylabel('Số lượng ảnh')
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                str(val), ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('data_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Đã lưu: data_distribution.png")

In [ ]:
import cv2
import random
from pathlib import Path
from PIL import Image
import numpy as np

def show_samples(data_dir, split='train', n_per_class=4):
    split_path = Path(data_dir) / split
    classes    = sorted([d.name for d in split_path.iterdir() if d.is_dir()])

    fig, axes = plt.subplots(len(classes), n_per_class,
                             figsize=(n_per_class * 3, len(classes) * 3))
    fig.suptitle(f'Mẫu ảnh — tập {split.upper()}', fontsize=14, fontweight='bold')

    for row, cls in enumerate(classes):
        cls_path = split_path / cls
        imgs = list(cls_path.glob('*.jpg')) + list(cls_path.glob('*.png'))
        samples = random.sample(imgs, min(n_per_class, len(imgs)))
        for col, img_path in enumerate(samples):
            img = cv2.imread(str(img_path))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            axes[row][col].imshow(img)
            axes[row][col].axis('off')
            if col == 0:
                axes[row][col].set_ylabel(cls, rotation=0, labelpad=80,
                                          fontsize=9, fontweight='bold')
    plt.tight_layout()
    plt.savefig('sample_images.png', dpi=120, bbox_inches='tight')
    plt.show()
    print("✅ Đã lưu: sample_images.png")

show_samples('/content/dataset', split='train', n_per_class=4)

In [ ]:
import sys
sys.path.append('/content/drive/MyDrive/face-mask-tracking-system/dataset/src')
from preprocess import ImagePreprocessor

preprocessor = ImagePreprocessor(target_size=(640, 640))

for split in ['train', 'val', 'test']:
    input_dir  = f'/content/dataset/{split}'
    output_dir = f'/content/dataset_processed/{split}'
    if os.path.exists(input_dir):
        print(f"\n▶ Đang xử lý tập {split}...")
        preprocessor.process_directory(input_dir, output_dir,
                                       min_blur_threshold=50)

In [ ]:
from preprocess import DataAugmentor

augmentor = DataAugmentor()

# Chỉ augment tập TRAIN (không augment val/test)
augmentor.augment_directory(
    input_dir   = '/content/dataset_processed/train',
    output_dir  = '/content/dataset_augmented/train',
    aug_per_image = 2
)
print("✅ Augmentation hoàn thành!")

In [ ]:
import os, random, torch
from collections import defaultdict, Counter
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Subset
from torchvision import transforms

def get_labels(dataset):
    '''Lấy labels từ dataset hoặc Subset'''
    if isinstance(dataset, torch.utils.data.Subset):
        return [dataset.dataset.targets[i] for i in dataset.indices]
    return dataset.targets

def limit_dataset_per_class(dataset, max_per_class=1750):
    '''Giới hạn số ảnh mỗi class để cân bằng dữ liệu'''
    class_indices = defaultdict(list)
    for idx, label in enumerate(dataset.targets):
        class_indices[label].append(idx)

    selected = []
    for label, indices in class_indices.items():
        if len(indices) > max_per_class:
            indices = random.sample(indices, max_per_class)
        selected.extend(indices)

    random.shuffle(selected)
    return Subset(dataset, selected)

def prepare_data(data_dir='/content/dataset_augmented',
                 batch_size=32, balance_train=True):
    '''Chuẩn bị DataLoader cho train/val/test'''
    transform = transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ])

    train_set = ImageFolder(os.path.join(data_dir, 'train'), transform=transform)
    val_set   = ImageFolder(os.path.join(data_dir, 'val'),   transform=transform)
    test_set  = ImageFolder(os.path.join(data_dir, 'test'),  transform=transform)

    print("Trước khi cân bằng:")
    print("  Train:", Counter(train_set.targets))
    print("  Val  :", Counter(val_set.targets))
    print("  Test :", Counter(test_set.targets))

    if balance_train:
        train_set = limit_dataset_per_class(train_set, max_per_class=1750)
        subset_labels = get_labels(train_set)
        print("\nSau khi cân bằng Train:")
        print("  Train:", Counter(subset_labels))

    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_set,   batch_size=batch_size, shuffle=False)
    test_loader  = DataLoader(test_set,  batch_size=batch_size, shuffle=False)

    return train_loader, val_loader, test_loader

# Chạy thử
train_loader, val_loader, test_loader = prepare_data()
print("\n✅ DataLoader sẵn sàng:")
print(f"   Train batches: {len(train_loader)}")
print(f"   Val   batches: {len(val_loader)}")
print(f"   Test  batches: {len(test_loader)}")

In [ ]:
from extract_frames import extract_frames

# extract_frames(
#     video_path    = '/content/drive/MyDrive/Face_Mask_Project/sample_video.mp4',
#     output_dir    = '/content/dataset/raw_frames',
#     frame_interval = 15   # lấy 1 frame mỗi 15 frames = 2 frames/giây (30fps)
# )
print("✅ (Bỏ comment cell này nếu có file video)")